# Comparación entre hilos y multiprocesamiento

Este notebook compara el tiempo de ejecución de tareas con hilos y procesos, y muestra el PID y el núcleo de CPU reportado durante cada cálculo.

Para ejecutarlo en Google Colab, selecciona **Entorno de ejecución > Ejecutar todas**.

## Celda 1: Instalación de dependencias

In [ ]:
!pip install psutil

## Celda 2: Comparación del tiempo de ejecución (Imagen 4)

Se ejecutan dos cálculos en hilos y luego en procesos. Los tiempos pueden variar según los recursos disponibles en la sesión de Colab.

In [ ]:
import time
import threading
import multiprocessing

def calc():
    s = 0
    for i in range(10_100_000):
        s += i
    return s

def tareas_hilos():
    hilo1 = threading.Thread(target=calc)
    hilo2 = threading.Thread(target=calc)
    hilo1.start(); hilo2.start()
    hilo1.join(); hilo2.join()

def tarea_multiprocessing():
    p1 = multiprocessing.Process(target=calc)
    p2 = multiprocessing.Process(target=calc)
    p1.start(); p2.start()
    p1.join(); p2.join()

inicio_ciclo_hilos = time.time()
tareas_hilos()
print("El tiempo de ejecución de los hilos", time.time() - inicio_ciclo_hilos)

inicio_ciclo_process = time.time()
tarea_multiprocessing()
print(f"El tiempo de ejecución de los procesos", time.time() - inicio_ciclo_process)

## Celda 3: Verificación de núcleos y GIL con `psutil` (Imágenes 1, 2 y 3)

`cpu_num()` informa el núcleo de CPU en el que se ejecutó el proceso en ese instante. En la prueba con hilos, ambos comparten el mismo proceso y están sujetos al GIL de CPython; en la prueba con multiprocesos, cada proceso tiene su propio intérprete.

In [ ]:
import time
import os
import psutil
import threading
import multiprocessing

def calc2():
    proc = psutil.Process(os.getpid())
    cpu = proc.cpu_num()
    print(f"[PID={os.getpid()}] | CPU={cpu} | Iniciando cálculo ...")
    total = 0
    for i in range(10_100_000):
        total += i
    print(f"[PID={os.getpid()}] | CPU={cpu} | Finalizando cálculo ...")
    return total

def prueba_hilos():
    hilo1 = threading.Thread(target=calc2)
    hilo2 = threading.Thread(target=calc2)
    hilo1.start(); hilo2.start()
    hilo1.join(); hilo2.join()

def prueba_multiprocesos():
    p1 = multiprocessing.Process(target=calc2)
    p2 = multiprocessing.Process(target=calc2)
    p1.start(); p2.start()
    p1.join(); p2.join()

print("--- Prueba con Hilos ---")
prueba_hilos()

print("\n--- Prueba con Multiprocesos ---")
prueba_multiprocesos()